In [1]:
%pip install pandas pyarrow duckdb ijson

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [1]:
from pathlib import Path
from collections import Counter
import json
import csv
import os
import re
import warnings

import pandas as pd
import pyarrow
import duckdb
import ijson

warnings.filterwarnings("default")

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 100)

print("Pandas:", pd.__version__)
print("PyArrow:", pyarrow.__version__)
print("DuckDB:", duckdb.__version__)
print("ijson:", ijson.__version__ if hasattr(ijson, "__version__") else "installed")

Pandas: 2.3.3
PyArrow: 21.0.0
DuckDB: 1.5.6
ijson: 3.6.0


In [2]:
cwd = Path.cwd().resolve()

if (cwd / "raw").exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / "raw").exists():
    PROJECT_ROOT = cwd.parent
else:
    raise FileNotFoundError(
        "Не найдена папка raw/. "
        "Проверь расположение ноутбука относительно корня проекта."
    )

RAW_DIR = PROJECT_ROOT / "raw"
PROCESSED_DIR = PROJECT_ROOT / "processed"
REPORTS_DIR = PROJECT_ROOT / "docs" / "data_quality"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print("Корень проекта:", PROJECT_ROOT)
print("RAW:", RAW_DIR)
print("Processed:", PROCESSED_DIR)
print("Отчёты:", REPORTS_DIR)

Корень проекта: C:\Users\245108\Desktop\gaming-data-analysis
RAW: C:\Users\245108\Desktop\gaming-data-analysis\raw
Processed: C:\Users\245108\Desktop\gaming-data-analysis\processed
Отчёты: C:\Users\245108\Desktop\gaming-data-analysis\docs\data_quality


In [3]:
def get_file_inventory(root: Path) -> pd.DataFrame:
    records = []

    for path in root.rglob("*"):
        if not path.is_file():
            continue

        try:
            size_bytes = path.stat().st_size

            records.append({
                "file": path.name,
                "relative_path": str(path.relative_to(root)),
                "extension": path.suffix.lower() or "[без расширения]",
                "size_bytes": size_bytes,
                "size_mb": round(size_bytes / 1024**2, 2),
                "size_gb": round(size_bytes / 1024**3, 3),
            })

        except OSError as exc:
            print(f"Не удалось прочитать метаданные {path}: {exc}")

    columns = [
        "file", "relative_path", "extension",
        "size_bytes", "size_mb", "size_gb"
    ]

    return pd.DataFrame(records, columns=columns)


inventory = get_file_inventory(RAW_DIR)

display(
    inventory.sort_values("size_bytes", ascending=False)
    .reset_index(drop=True)
)

print(f"Всего файлов: {len(inventory):,}")
print(f"Общий объём: {inventory['size_gb'].sum():.2f} ГБ")

print("\nКоличество файлов по расширениям:")
display(
    inventory.groupby("extension")
    .agg(
        files=("file", "count"),
        total_mb=("size_mb", "sum"),
    )
    .sort_values("total_mb", ascending=False)
)

,file,relative_path,extension,size_bytes,size_mb,size_gb
0,videogames_data.jsonl,rawg\jsonl\videogames_data.jsonl,.jsonl,4300705739,4101.47,4.005
1,dataset.csv,steam\reviews\dataset.csv,.csv,2160871419,2060.77,2.012
2,recommendations.csv,steam\recommendations\recommendations.csv,.csv,2023223067,1929.50,1.884
3,videogames_data.csv,rawg\videogames_data.csv,.csv,2000469667,1907.80,1.863
4,videogames_data.parquet,rawg\videogames_data.parquet,.parquet,1011662649,964.80,0.942
5,games.json,steam\fronkongames\games.json,.json,970222554,925.28,0.904
6,reviews.csv,steam\games_2025\steam_dataset_2025_csv\reviews.csv,.csv,669812331,638.78,0.624
7,games_march2025_full.csv,steam\games_artemiloff\games_march2025_full.csv,.csv,471492150,449.65,0.439
8,games_march2025_cleaned.csv,steam\games_artemiloff\games_march2025_cleaned.csv,.csv,468641107,446.93,0.436
9,games_may2024_full.csv,steam\games_artemiloff\games_may2024_full.csv,.csv,424919226,405.23,0.396


Всего файлов: 42
Общий объём: 14.76 ГБ

Количество файлов по расширениям:


,files,total_mb
extension,,
.csv,36,9101.71
.jsonl,1,4101.47
.parquet,1,964.80
.json,3,942.02
.py,1,0.01


In [4]:
inventory.to_csv(
    REPORTS_DIR / "raw_file_inventory.csv",
    index=False,
    encoding="utf-8-sig",
)

print("Инвентаризация сохранена.")

Инвентаризация сохранена.


In [5]:
EXPECTED_SOURCES = {
    "rawg_games": "rawg",
    "steam_games_fronkongames": "steam/fronkongames",
    "steam_reviews": "steam/reviews",
    "steam_recommendations": "steam/recommendations",
    "steam_games_2025": "steam/games_2025",
    "steam_games_2021_2025": "steam/games_2021_2025",
    "steam_games_artemiloff": "steam/games_artemiloff",
    "steam_monthly_players": "steam/monthly_players",
    "metacritic_vs_steam": "steam/metacritic_vs_steam",
    "ultimate_games": "games/ultimate_games",
    "twitch_streamers": "twitch/streamers",
    "twitch_evolution": "twitch/evolution",
    "epic_games": "epic",
}

source_check = []

for source, relative_dir in EXPECTED_SOURCES.items():
    source_path = RAW_DIR / relative_dir

    files = [
        path for path in source_path.rglob("*")
        if path.is_file()
    ] if source_path.exists() else []

    source_check.append({
        "source": source,
        "expected_directory": relative_dir,
        "directory_exists": source_path.exists(),
        "file_count": len(files),
        "total_mb": round(
            sum(path.stat().st_size for path in files) / 1024**2, 2
        ),
    })

source_check = pd.DataFrame(source_check)

display(source_check)

print(
    "Источники без найденных файлов:",
    source_check.loc[
        source_check["file_count"] == 0, "source"
    ].tolist()
)

,source,expected_directory,directory_exists,file_count,total_mb
0,rawg_games,rawg,True,3,6974.07
1,steam_games_fronkongames,steam/fronkongames,True,2,1307.83
2,steam_reviews,steam/reviews,True,1,2060.77
3,steam_recommendations,steam/recommendations,True,4,2134.99
4,steam_games_2025,steam/games_2025,True,13,841.52
5,steam_games_2021_2025,steam/games_2021_2025,True,2,11.12
6,steam_games_artemiloff,steam/games_artemiloff,True,4,1704.57
7,steam_monthly_players,steam/monthly_players,True,1,36.13
8,metacritic_vs_steam,steam/metacritic_vs_steam,True,4,9.51
9,ultimate_games,games/ultimate_games,True,1,27.54


Источники без найденных файлов: []


## Инвентаризация по csv

In [24]:
def inspect_csv_header(path: Path) -> dict:
    result = {
        "file": path.name,
        "relative_path": str(path.relative_to(RAW_DIR)),
        "columns_count": None,
        "columns": None,
        "encoding": None,
        "delimiter": None,
        "error": None,
    }

    for encoding in ("utf-8-sig", "utf-8", "cp1252", "latin1"):
        try:
            with path.open(
                "r", encoding=encoding, newline=""
            ) as file:
                sample = file.read(65536)

                if not sample.strip():
                    result["error"] = "Пустой файл"
                    return result

                try:
                    dialect = csv.Sniffer().sniff(
                        sample, delimiters=",;\t|"
                    )
                    delimiter = dialect.delimiter
                except csv.Error:
                    delimiter = ","

                reader = csv.reader(
                    [sample.splitlines()[0]],
                    delimiter=delimiter,
                )
                columns = next(reader)

                result.update({
                    "columns_count": len(columns),
                    "columns": columns,
                    "encoding": encoding,
                    "delimiter": repr(delimiter),
                })

                return result

        except UnicodeDecodeError:
            continue
        except Exception as exc:
            result["error"] = str(exc)
            return result

    result["error"] = "Не удалось определить кодировку"
    return result


csv_files = [
    RAW_DIR / relative
    for relative in inventory.loc[
        inventory["extension"] == ".csv", "relative_path"
    ]
]

csv_schema_records = [
    inspect_csv_header(path)
    for path in csv_files
]

csv_schemas = pd.DataFrame(csv_schema_records)

display(
    csv_schemas[
        ["relative_path", "columns_count", "columns",
         "encoding", "delimiter", "error"]
    ]
)

,relative_path,columns_count,columns,encoding,delimiter,error
0,epic\epic games store-video games.csv,17,"[game_link, name, price of game, developer of game, genres of games, features of game, publisher...",utf-8-sig,"','",None
1,epic\epic-games-apps.csv,16,"[links, name, type, genres, features, price, developer, publisher, release_date, platform, os re...",utf-8-sig,"','",None
2,epic\epic-games-demo.csv,17,"[links, name, type, genres, features, Developer, publisher, release date, initial release in win...",utf-8-sig,"','",None
3,epic\epic_games_editor.csv,20,"[links, name, type, genres, features, Developer, publisher, release date, platform, os recommend...",utf-8-sig,"','",None
4,rawg\videogames_data.csv,54,"[id, slug, name, released, background_image, rating, rating_top, ratings_count, reviews_text_cou...",utf-8-sig,"','",None
5,twitch\evolution\Twitch_game_data.csv,12,"[Rank, Game, Month, Year, Hours_watched, Hours_streamed, Peak_viewers, Peak_channels, Streamers,...",utf-8-sig,"','",None
6,twitch\evolution\Twitch_global_data.csv,9,"[year, Month, Hours_watched, Avg_viewers, Peak_viewers, Streams, Avg_channels, Games_streamed, V...",utf-8-sig,"','",None
7,twitch\streamers\datasetV2.csv,17,"[RANK, NAME, LANGUAGE, TYPE, MOST_STREAMED_GAME, 2ND_MOST_STREAMED_GAME, AVERAGE_STREAM_DURATION...",utf-8-sig,"','",None
8,steam\fronkongames\games.csv,39,"[AppID, Name, Release date, Estimated owners, Peak CCU, Required age, Price, DiscountDLC count, ...",utf-8-sig,"','",None
9,steam\games_2021_2025\a_steam_data_2021_2025.csv,10,"[appid, name, release_year, release_date, genres, categories, price, recommendations, developer,...",utf-8-sig,"','",None


## Инвентаризация по JSON

In [7]:
def find_file_by_name(root: Path, filename: str):
    matches = list(root.rglob(filename))

    if not matches:
        print(f"Файл не найден: {filename}")
        return None

    if len(matches) > 1:
        print(f"Найдено несколько файлов {filename}:")
        for path in matches:
            print(" ", path.relative_to(root))

    return matches[0]


games_json_path = find_file_by_name(RAW_DIR, "games.json")
rawg_jsonl_path = find_file_by_name(RAW_DIR, "videogames_data.jsonl")


def inspect_file_prefix(path: Path, n_bytes=300):
    if path is None:
        return

    with path.open("rb") as file:
        prefix = file.read(n_bytes)

    print(f"\nФайл: {path.relative_to(RAW_DIR)}")
    print("Размер, МБ:", round(path.stat().st_size / 1024**2, 2))
    print("Начало файла:")
    print(prefix.decode("utf-8", errors="replace"))


inspect_file_prefix(games_json_path)
inspect_file_prefix(rawg_jsonl_path)


Файл: steam\fronkongames\games.json
Размер, МБ: 925.28
Начало файла:
{
    "2539430": {
        "name": "Black Dragon Mage Playtest",
        "release_date": "Aug 1, 2023",
        "required_age": 0,
        "price": 0.0,
        "dlc_count": 0,
        "detailed_description": "",
        "about_the_game": "",
        "short_description": "",
        "reviews": "",
 

Файл: rawg\jsonl\videogames_data.jsonl
Размер, МБ: 4101.47
Начало файла:
{"id":3498,"slug":"grand-theft-auto-v","name":"Grand Theft Auto V","released":"2013-09-17","background_image":"https://media.rawg.io/media/games/20a/20aa03a10cda45239fe22d035c0ebe64.jpg","rating":4.47,"rating_top":5,"ratings_count":7409,"reviews_text_count":135,"added":22635,"metacritic":92,"playtim


In [8]:
if games_json_path is not None:
    with games_json_path.open("rb") as file:
        app_id, game = next(ijson.kvitems(file, ""))

    print("Пример AppID:", app_id)
    print("Тип записи:", type(game).__name__)
    print("Число полей:", len(game))
    print("Поля:", list(game.keys()))

Пример AppID: 2539430
Тип записи: dict
Число полей: 42
Поля: ['name', 'release_date', 'required_age', 'price', 'dlc_count', 'detailed_description', 'about_the_game', 'short_description', 'reviews', 'header_image', 'website', 'support_url', 'support_email', 'windows', 'mac', 'linux', 'metacritic_score', 'metacritic_url', 'achievements', 'recommendations', 'notes', 'supported_languages', 'full_audio_languages', 'packages', 'developers', 'publishers', 'categories', 'genres', 'screenshots', 'movies', 'user_score', 'score_rank', 'positive', 'negative', 'estimated_owners', 'average_playtime_forever', 'average_playtime_2weeks', 'median_playtime_forever', 'median_playtime_2weeks', 'discount', 'peak_ccu', 'tags']


In [9]:
if rawg_jsonl_path is not None:
    with rawg_jsonl_path.open(
        "r", encoding="utf-8"
    ) as file:
        for i, line in enumerate(file):
            if not line.strip():
                continue

            try:
                record = json.loads(line)
                print(f"\nЗапись {i + 1}")
                print("Тип:", type(record).__name__)

                if isinstance(record, dict):
                    print("Поля:", list(record.keys())[:50])
                else:
                    print("Пример:", str(record)[:300])

            except json.JSONDecodeError as exc:
                print("Ошибка JSON в строке", i + 1, exc)

            if i >= 2:
                break


Запись 1
Тип: dict
Поля: ['id', 'slug', 'name', 'released', 'background_image', 'rating', 'rating_top', 'ratings_count', 'reviews_text_count', 'added', 'metacritic', 'playtime', 'suggestions_count', 'updated', 'reviews_count', 'saturated_color', 'dominant_color', 'platforms', 'stores', 'developers', 'genres', 'tags', 'publishers', 'esrb_rating', 'added_by_status', 'metacritic_url', 'ratings', 'clip', 'name_original', 'reddit_url', 'movies_count', 'reactions', 'background_image_additional', 'website', 'reddit_count', 'achievements_count', 'youtube_count', 'short_screenshots', 'creators_count', 'twitch_count', 'alternative_names', 'parent_achievements_count', 'additions_count', 'metacritic_platforms', 'screenshots_count', 'parent_platforms', 'game_series_count', 'description', 'description_raw']

Запись 2
Тип: dict
Поля: ['id', 'slug', 'name', 'released', 'background_image', 'rating', 'rating_top', 'ratings_count', 'reviews_text_count', 'added', 'metacritic', 'playtime', 'suggestions_co

In [10]:
csv_schemas.to_csv(
    REPORTS_DIR / "csv_schemas.csv",
    index=False,
    encoding="utf-8-sig",
)

source_check.to_csv(
    REPORTS_DIR / "source_check.csv",
    index=False,
    encoding="utf-8-sig",
)

print("Сохранены:")
print("-", REPORTS_DIR / "raw_file_inventory.csv")
print("-", REPORTS_DIR / "csv_schemas.csv")
print("-", REPORTS_DIR / "source_check.csv")

Сохранены:
- C:\Users\245108\Desktop\gaming-data-analysis\docs\data_quality\raw_file_inventory.csv
- C:\Users\245108\Desktop\gaming-data-analysis\docs\data_quality\csv_schemas.csv
- C:\Users\245108\Desktop\gaming-data-analysis\docs\data_quality\source_check.csv


## Дмагностика качества строк

In [11]:
import csv
from pathlib import Path
import pandas as pd


def check_csv_structure(path: Path, max_records: int = 1000) -> dict:
    result = {
        "file": str(path.relative_to(RAW_DIR)),
        "header_columns": None,
        "records_checked": 0,
        "bad_records": 0,
        "examples": [],
        "error": None,
    }

    for encoding in ("utf-8-sig", "utf-8", "cp1252", "latin1"):
        try:
            with path.open(
                "r", encoding=encoding, newline=""
            ) as file:
                reader = csv.reader(file)

                header = next(reader, None)

                if header is None:
                    result["error"] = "Пустой файл"
                    return result

                result["header_columns"] = len(header)

                for row_number, row in enumerate(
                    reader, start=2
                ):
                    result["records_checked"] += 1

                    if len(row) != len(header):
                        result["bad_records"] += 1

                        if len(result["examples"]) < 5:
                            result["examples"].append({
                                "row": row_number,
                                "expected_fields": len(header),
                                "actual_fields": len(row),
                            })

                    if result["records_checked"] >= max_records:
                        break

            return result

        except UnicodeDecodeError:
            continue
        except Exception as exc:
            result["error"] = str(exc)
            return result

    result["error"] = "Не удалось определить кодировку"
    return result


# Сначала проверяем файлы, которые важнее всего для анализа.
priority_files = [
    "steam/fronkongames/games.csv",
    "steam/artemiloff/games_march2025_full.csv",
    "steam/artemiloff/games_march2025_cleaned.csv",
    "steam/artemiloff/games_may2024_full.csv",
    "steam/artemiloff/games_may2024_cleaned.csv",
    "epic/epic games store-video games.csv",
    "epic/epic-games-apps.csv",
    "epic/epic-games-demo.csv",
    "epic/epic_games_editor.csv",
]

structure_results = []

for relative_path in priority_files:
    path = RAW_DIR / relative_path

    if path.exists():
        structure_results.append(
            check_csv_structure(path)
        )
    else:
        print(f"Файл не найден: {relative_path}")

structure_df = pd.DataFrame(structure_results)

display(structure_df)

Файл не найден: steam/artemiloff/games_march2025_full.csv
Файл не найден: steam/artemiloff/games_march2025_cleaned.csv
Файл не найден: steam/artemiloff/games_may2024_full.csv
Файл не найден: steam/artemiloff/games_may2024_cleaned.csv


,file,header_columns,records_checked,bad_records,examples,error
0,steam\fronkongames\games.csv,39,1000,1000,"[{'row': 2, 'expected_fields': 39, 'actual_fields': 40}, {'row': 3, 'expected_fields': 39, 'actu...",None
1,epic\epic games store-video games.csv,17,899,0,[],None
2,epic\epic-games-apps.csv,16,11,0,[],None
3,epic\epic-games-demo.csv,17,25,0,[],None
4,epic\epic_games_editor.csv,20,20,0,[],None


In [12]:
import pandas as pd

epic_files = [
    "epic games store-video games.csv",
    "epic-games-apps.csv",
    "epic-games-demo.csv",
    "epic_games_editor.csv",
]

epic_frames = {}
epic_summary = []

for filename in epic_files:
    path = RAW_DIR / "epic" / filename

    if not path.exists():
        continue

    df = pd.read_csv(
        path,
        encoding="utf-8-sig",
        nrows=100_000,
        low_memory=False,
    )

    # Для сравнения используем нормализованное название.
    name_col = next(
        (col for col in df.columns if col.lower() == "name"),
        None,
    )

    if name_col is None:
        print(f"Нет колонки name: {filename}")
        continue

    df["_normalized_name"] = (
        df[name_col]
        .astype("string")
        .str.strip()
        .str.casefold()
    )

    epic_frames[filename] = df

    epic_summary.append({
        "file": filename,
        "rows_loaded": len(df),
        "unique_names": df["_normalized_name"].nunique(),
        "duplicate_names": int(
            df["_normalized_name"].duplicated().sum()
        ),
    })

display(pd.DataFrame(epic_summary))

,file,rows_loaded,unique_names,duplicate_names
0,epic games store-video games.csv,899,899,0
1,epic-games-apps.csv,11,11,0
2,epic-games-demo.csv,25,21,4
3,epic_games_editor.csv,20,18,1


In [13]:
epic_name_sets = {
    filename: set(
        df["_normalized_name"].dropna()
    )
    for filename, df in epic_frames.items()
}

overlap_records = []

for i, (file_a, names_a) in enumerate(epic_name_sets.items()):
    for file_b, names_b in list(epic_name_sets.items())[i + 1:]:
        common = names_a & names_b

        overlap_records.append({
            "file_a": file_a,
            "file_b": file_b,
            "common_names": len(common),
            "share_of_a_pct": round(
                100 * len(common) / len(names_a), 2
            ) if names_a else 0,
            "share_of_b_pct": round(
                100 * len(common) / len(names_b), 2
            ) if names_b else 0,
            "examples": list(sorted(common))[:5],
        })

display(
    pd.DataFrame(overlap_records).sort_values(
        "common_names", ascending=False
    )
)

,file_a,file_b,common_names,share_of_a_pct,share_of_b_pct,examples
1,epic games store-video games.csv,epic-games-demo.csv,3,0.33,14.29,"[bonfire peaks, ghost recon breakpoint, startup panic]"
0,epic games store-video games.csv,epic-games-apps.csv,0,0.00,0.00,[]
2,epic games store-video games.csv,epic_games_editor.csv,0,0.00,0.00,[]
3,epic-games-apps.csv,epic-games-demo.csv,0,0.00,0.00,[]
4,epic-games-apps.csv,epic_games_editor.csv,0,0.00,0.00,[]
5,epic-games-demo.csv,epic_games_editor.csv,0,0.00,0.00,[]


In [14]:
import ijson
from collections import Counter

games_json_path = RAW_DIR / "steam" / "fronkongames" / "games.json"

field_total = Counter()
field_missing = Counter()
game_count = 0

def is_missing_value(value):
    return (
        value is None
        or (isinstance(value, str) and not value.strip())
    )

with games_json_path.open("rb") as file:
    for app_id, game in ijson.kvitems(file, ""):
        game_count += 1

        for field, value in game.items():
            field_total[field] += 1

            if is_missing_value(value):
                field_missing[field] += 1

missing_records = []

for field, total in field_total.items():
    missing = field_missing[field]

    missing_records.append({
        "field": field,
        "records": total,
        "missing": missing,
        "missing_pct": round(
            100 * missing / total, 2
        ) if total else 0,
    })

steam_missing_report = (
    pd.DataFrame(missing_records)
    .sort_values("missing_pct", ascending=False)
    .reset_index(drop=True)
)

print(f"Всего обработано игр: {game_count:,}")
display(steam_missing_report)

steam_missing_report.to_csv(
    REPORTS_DIR / "steam_json_missing_values.csv",
    index=False,
    encoding="utf-8-sig",
)

Всего обработано игр: 144,499


,field,records,missing,missing_pct
0,score_rank,144499,144459,99.97
1,metacritic_url,144499,140217,97.04
2,reviews,144499,131850,91.25
3,notes,144499,116973,80.95
4,website,144499,88860,61.50
5,support_url,144499,82534,57.12
6,support_email,144499,24607,17.03
7,about_the_game,144499,8524,5.90
8,detailed_description,144499,8492,5.88
9,short_description,144499,8333,5.77


## l

In [15]:

from pathlib import Path
import json
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import ijson

PROCESSED_DIR = PROJECT_ROOT / "processed"
REPORTS_DIR = PROJECT_ROOT / "docs" / "data_quality"

OUTPUT_DIRS = [
    "games",
    "reviews",
    "recommendations",
    "players",
    "twitch",
    "youtube",
]

for directory in OUTPUT_DIRS:
    (PROCESSED_DIR / directory).mkdir(
        parents=True,
        exist_ok=True,
    )

REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print("Подготовлены каталоги processed/ и docs/data_quality/")


Подготовлены каталоги processed/ и docs/data_quality/


In [16]:

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import ijson
import json

steam_json_path = (
    RAW_DIR / "steam" / "fronkongames" / "games.json"
)

steam_parquet_path = (
    PROCESSED_DIR / "games" / "steam_games.parquet"
)

if not steam_json_path.exists():
    raise FileNotFoundError(steam_json_path)

# Числовые поля исходного каталога Steam.
STEAM_INTEGER_FIELDS = [
    "required_age",
    "dlc_count",
    "achievements",
    "recommendations",
    "metacritic_score",
    "score_rank",
    "positive",
    "negative",
    "average_playtime_forever",
    "average_playtime_2weeks",
    "median_playtime_forever",
    "median_playtime_2weeks",
    "peak_ccu",
]

STEAM_FLOAT_FIELDS = [
    "price",
    "user_score",
    "discount",
]

STEAM_BOOLEAN_FIELDS = [
    "windows",
    "mac",
    "linux",
]


import json
from decimal import Decimal


def json_default(value):
    if isinstance(value, Decimal):
        if value == value.to_integral_value():
            return int(value)
        return float(value)

    raise TypeError(
        f"Object of type {type(value).__name__} is not JSON serializable"
    )


def serialize_nested_values(record):
    result = {}

    for key, value in record.items():
        if isinstance(value, (dict, list)):
            result[key] = json.dumps(
                value,
                ensure_ascii=False,
                default=json_default,
            )
        elif isinstance(value, Decimal):
            result[key] = json_default(value)
        else:
            result[key] = value

    return result


def prepare_steam_batch(records):
    df = pd.DataFrame(records)

    # Фиксируем типы независимо от состава конкретного пакета.
    for col in STEAM_INTEGER_FIELDS:
        if col in df.columns:
            df[col] = pd.to_numeric(
                df[col], errors="coerce"
            ).astype("Int64")

    for col in STEAM_FLOAT_FIELDS:
        if col in df.columns:
            df[col] = pd.to_numeric(
                df[col], errors="coerce"
            ).astype("Float64")

    for col in STEAM_BOOLEAN_FIELDS:
        if col in df.columns:
            df[col] = df[col].map({
                True: True,
                False: False,
                1: True,
                0: False,
                "true": True,
                "false": False,
            }).astype("boolean")

    # Текстовые поля приводим к единому типу.
    typed_columns = (
        set(STEAM_INTEGER_FIELDS)
        | set(STEAM_FLOAT_FIELDS)
        | set(STEAM_BOOLEAN_FIELDS)
    )

    for col in df.columns:
        if col not in typed_columns:
            df[col] = df[col].astype("string")

    return df


BATCH_SIZE = 5_000
batch = []
writer = None
total_rows = 0

try:
    with steam_json_path.open("rb") as file:
        for appid, game in ijson.kvitems(file, ""):
            record = serialize_nested_values(game)
            record["appid"] = appid
            batch.append(record)

            if len(batch) >= BATCH_SIZE:
                df = prepare_steam_batch(batch)
                table = pa.Table.from_pandas(
                    df,
                    preserve_index=False,
                )

                if writer is None:
                    writer = pq.ParquetWriter(
                        steam_parquet_path,
                        table.schema,
                        compression="zstd",
                    )

                table = table.cast(writer.schema)
                writer.write_table(table)

                total_rows += len(df)
                batch.clear()

                print(f"Обработано: {total_rows:,}")

    if batch:
        df = prepare_steam_batch(batch)
        table = pa.Table.from_pandas(
            df,
            preserve_index=False,
        )

        if writer is None:
            writer = pq.ParquetWriter(
                steam_parquet_path,
                table.schema,
                compression="zstd",
            )

        table = table.cast(writer.schema)
        writer.write_table(table)
        total_rows += len(df)

finally:
    if writer is not None:
        writer.close()

print(f"Итого обработано игр: {total_rows:,}")
print(f"Результат: {steam_parquet_path}")


Обработано: 5,000
Обработано: 10,000
Обработано: 15,000
Обработано: 20,000
Обработано: 25,000
Обработано: 30,000
Обработано: 35,000
Обработано: 40,000
Обработано: 45,000
Обработано: 50,000
Обработано: 55,000
Обработано: 60,000
Обработано: 65,000
Обработано: 70,000
Обработано: 75,000
Обработано: 80,000
Обработано: 85,000
Обработано: 90,000
Обработано: 95,000
Обработано: 100,000
Обработано: 105,000
Обработано: 110,000
Обработано: 115,000
Обработано: 120,000
Обработано: 125,000
Обработано: 130,000
Обработано: 135,000
Обработано: 140,000
Итого обработано игр: 144,499
Результат: C:\Users\245108\Desktop\gaming-data-analysis\processed\games\steam_games.parquet


In [17]:

RAWG_COLUMNS = [
    "id", "slug", "name", "released", "background_image",
    "rating", "rating_top", "ratings_count",
    "reviews_text_count", "added", "metacritic", "playtime",
    "suggestions_count", "updated", "reviews_count",
    "saturated_color", "dominant_color", "platforms", "stores",
    "developers", "genres", "tags", "publishers", "esrb_rating",
    "added_by_status", "metacritic_url", "ratings", "clip",
    "name_original", "reddit_url", "movies_count", "reactions",
    "background_image_additional", "website", "reddit_count",
    "achievements_count", "youtube_count", "short_screenshots",
    "creators_count", "twitch_count", "alternative_names",
    "parent_achievements_count", "additions_count",
    "metacritic_platforms", "screenshots_count",
    "parent_platforms", "game_series_count", "description",
    "description_raw",
]


In [18]:

from pathlib import Path
import json
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

rawg_jsonl_path = (
    RAW_DIR / "rawg" / "jsonl" / "videogames_data.jsonl"
)

rawg_parquet_path = (
    PROCESSED_DIR / "games" / "rawg_games.parquet"
)

RAWG_INTEGER_FIELDS = [
    "id",
    "rating_top",
    "ratings_count",
    "reviews_text_count",
    "added",
    "metacritic",
    "suggestions_count",
    "reviews_count",
    "movies_count",
    "reddit_count",
    "achievements_count",
    "youtube_count",
    "creators_count",
    "twitch_count",
    "parent_achievements_count",
    "additions_count",
    "screenshots_count",
    "game_series_count",
]

RAWG_FLOAT_FIELDS = ["rating", "playtime"]

def prepare_rawg_batch(df):
    df = df.reindex(columns=RAWG_COLUMNS)

    # Вложенные объекты храним в JSON-строках.
    for col in df.columns:
        if df[col].dtype == "object":
            df[col] = df[col].map(
                lambda value: json.dumps(
                    value, ensure_ascii=False
                )
                if isinstance(value, (dict, list))
                else value
            )

    for col in RAWG_INTEGER_FIELDS:
        if col in df.columns:
            df[col] = pd.to_numeric(
                df[col], errors="coerce"
            ).astype("Int64")

    for col in RAWG_FLOAT_FIELDS:
        if col in df.columns:
            df[col] = pd.to_numeric(
                df[col], errors="coerce"
            ).astype("Float64")

    typed_columns = (
        set(RAWG_INTEGER_FIELDS)
        | set(RAWG_FLOAT_FIELDS)
    )

    for col in df.columns:
        if col not in typed_columns:
            df[col] = df[col].astype("string")

    return df


if not rawg_jsonl_path.exists():
    raise FileNotFoundError(rawg_jsonl_path)

BATCH_SIZE = 20_000
writer = None
total_rows = 0

try:
    for chunk in pd.read_json(
        rawg_jsonl_path,
        lines=True,
        chunksize=BATCH_SIZE,
        encoding="utf-8",
    ):
        chunk = prepare_rawg_batch(chunk)

        table = pa.Table.from_pandas(
            chunk,
            preserve_index=False,
        )

        if writer is None:
            writer = pq.ParquetWriter(
                rawg_parquet_path,
                table.schema,
                compression="zstd",
            )

        table = table.select(writer.schema.names)
        table = table.cast(writer.schema)
        writer.write_table(table)

        total_rows += len(chunk)
        print(f"Обработано RAWG-записей: {total_rows:,}")

finally:
    if writer is not None:
        writer.close()

print(f"Итого: {total_rows:,}")
print(f"Результат: {rawg_parquet_path}")


Обработано RAWG-записей: 20,000
Обработано RAWG-записей: 40,000
Обработано RAWG-записей: 60,000
Обработано RAWG-записей: 80,000
Обработано RAWG-записей: 100,000
Обработано RAWG-записей: 120,000
Обработано RAWG-записей: 140,000
Обработано RAWG-записей: 160,000
Обработано RAWG-записей: 180,000
Обработано RAWG-записей: 200,000
Обработано RAWG-записей: 220,000
Обработано RAWG-записей: 240,000
Обработано RAWG-записей: 260,000
Обработано RAWG-записей: 280,000
Обработано RAWG-записей: 300,000
Обработано RAWG-записей: 320,000
Обработано RAWG-записей: 340,000
Обработано RAWG-записей: 360,000
Обработано RAWG-записей: 380,000
Обработано RAWG-записей: 400,000
Обработано RAWG-записей: 420,000
Обработано RAWG-записей: 440,000
Обработано RAWG-записей: 460,000
Обработано RAWG-записей: 480,000
Обработано RAWG-записей: 500,000
Обработано RAWG-записей: 520,000
Обработано RAWG-записей: 540,000
Обработано RAWG-записей: 560,000
Обработано RAWG-записей: 580,000
Обработано RAWG-записей: 600,000
Обработано RAW

In [19]:

import duckdb
import pandas as pd

con = duckdb.connect()

PARQUET_FILES = {
    "steam_games": (
        PROCESSED_DIR / "games" / "steam_games.parquet"
    ),
    "rawg_games": (
        PROCESSED_DIR / "games" / "rawg_games.parquet"
    ),
}

quality_summary = []

for dataset_name, path in PARQUET_FILES.items():
    if not path.exists():
        print(f"Пропуск: файл не найден — {path}")
        continue

    path_sql = str(path).replace("\\", "/").replace("'", "''")

    query = f"""
        SELECT
            COUNT(*) AS row_count,
            COUNT(*) - COUNT(*) FILTER (
                WHERE name IS NOT NULL AND TRIM(name) <> ''
            ) AS missing_name,
            COUNT(DISTINCT name) AS unique_names
        FROM read_parquet('{path_sql}')
    """

    result = con.execute(query).fetchone()

    quality_summary.append({
        "dataset": dataset_name,
        "file": str(path.relative_to(PROJECT_ROOT)),
        "row_count": result[0],
        "missing_name": result[1],
        "unique_names": result[2],
    })

quality_df = pd.DataFrame(quality_summary)
display(quality_df)

quality_df.to_csv(
    REPORTS_DIR / "prepared_games_quality.csv",
    index=False,
    encoding="utf-8-sig",
)


,dataset,file,row_count,missing_name,unique_names
0,steam_games,processed\games\steam_games.parquet,144499,1,143026
1,rawg_games,processed\games\rawg_games.parquet,899585,0,899585


In [20]:

for dataset_name, path in PARQUET_FILES.items():
    if not path.exists():
        print(f"NOT FOUND: {dataset_name}")
        continue

    table = pq.read_schema(path)

    print(f"\n{'=' * 60}")
    print(dataset_name)
    print(f"Размер: {path.stat().st_size / 1024**2:.2f} MB")
    print(table)



steam_games
Размер: 234.56 MB
name: string
release_date: string
required_age: int64
price: double
dlc_count: int64
detailed_description: string
about_the_game: string
short_description: string
reviews: string
header_image: string
website: string
support_url: string
support_email: string
windows: bool
mac: bool
linux: bool
metacritic_score: int64
metacritic_url: string
achievements: int64
recommendations: int64
notes: string
supported_languages: string
full_audio_languages: string
packages: string
developers: string
publishers: string
categories: string
genres: string
screenshots: string
movies: string
user_score: double
score_rank: int64
positive: int64
negative: int64
estimated_owners: string
average_playtime_forever: int64
average_playtime_2weeks: int64
median_playtime_forever: int64
median_playtime_2weeks: int64
discount: double
peak_ccu: int64
tags: string
appid: string
-- schema metadata --
pandas: '{"index_columns": [], "column_indexes": [], "columns": [{"name":' + 5311

rawg_ga

## 

In [21]:

def csv_to_parquet(csv_path: Path, parquet_path: Path):
    """Преобразует проверенный CSV в Parquet через DuckDB."""
    csv_path = Path(csv_path)
    parquet_path = Path(parquet_path)

    if not csv_path.exists():
        raise FileNotFoundError(csv_path)

    parquet_path.parent.mkdir(parents=True, exist_ok=True)

    # Экранируем пути для SQL.
    source_sql = str(csv_path.resolve()).replace("\\", "/").replace("'", "''")
    target_sql = str(parquet_path.resolve()).replace("\\", "/").replace("'", "''")

    query = f"""
        COPY (
            SELECT *
            FROM read_csv_auto(
                '{source_sql}',
                header = true,
                sample_size = -1,
                ignore_errors = false
            )
        )
        TO '{target_sql}'
        (FORMAT PARQUET, COMPRESSION ZSTD)
    """

    con.execute(query)

    # Контроль результата.
    row_count = con.execute(
        f"SELECT COUNT(*) FROM read_parquet('{target_sql}')"
    ).fetchone()[0]

    print(f"Источник: {csv_path.name}")
    print(f"Результат: {parquet_path}")
    print(f"Записей: {row_count:,}")

    return row_count


In [26]:
csv_to_parquet(
    RAW_DIR / "epic" / "epic games store-video games.csv",
    PROCESSED_DIR / "games" / "epic_games.parquet",
)

csv_to_parquet(
    RAW_DIR / "steam" / "monthly_players" / "steamcharts.csv",
    PROCESSED_DIR / "players" / "steam_monthly_players.parquet",
)

Источник: epic games store-video games.csv
Результат: C:\Users\245108\Desktop\gaming-data-analysis\processed\games\epic_games.parquet
Записей: 899
Источник: steamcharts.csv
Результат: C:\Users\245108\Desktop\gaming-data-analysis\processed\players\steam_monthly_players.parquet
Записей: 612,265


612265

In [29]:

def csv_to_parquet(csv_path: Path, parquet_path: Path):
    """Преобразует проверенный CSV в Parquet через DuckDB."""
    csv_path = Path(csv_path)
    parquet_path = Path(parquet_path)

    if not csv_path.exists():
        raise FileNotFoundError(csv_path)

    parquet_path.parent.mkdir(parents=True, exist_ok=True)

    # Экранируем пути для SQL.
    source_sql = str(csv_path.resolve()).replace("\\", "/").replace("'", "''")
    target_sql = str(parquet_path.resolve()).replace("\\", "/").replace("'", "''")

    query = f"""
        COPY (
            SELECT *
            FROM read_csv_auto(
    '{source_sql}',
    header=true,
    sample_size=-1,
    ignore_errors=false,
    encoding='cp1250'
                )
            )
        
        TO '{target_sql}'
        (FORMAT PARQUET, COMPRESSION ZSTD)
    """

    con.execute(query)

    # Контроль результата.
    row_count = con.execute(
        f"SELECT COUNT(*) FROM read_parquet('{target_sql}')"
    ).fetchone()[0]

    print(f"Источник: {csv_path.name}")
    print(f"Результат: {parquet_path}")
    print(f"Записей: {row_count:,}")

    return row_count


In [30]:
csv_to_parquet(
    RAW_DIR / "twitch" / "evolution" / "Twitch_game_data.csv",
    PROCESSED_DIR / "twitch" / "twitch_games.parquet",
)

Источник: Twitch_game_data.csv
Результат: C:\Users\245108\Desktop\gaming-data-analysis\processed\twitch\twitch_games.parquet
Записей: 21,000


21000

## Итоговая проверка

In [31]:

from pathlib import Path
import pandas as pd
import pyarrow.parquet as pq

# Если эти переменные уже объявлены выше, повторно менять их не нужно.
PROJECT_ROOT = Path.cwd()

# Если ноутбук находится в папке notebooks/, поднимаемся в корень.
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

PROCESSED_DIR = PROJECT_ROOT / "processed"
REPORTS_DIR = PROJECT_ROOT / "docs" / "data_quality"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

inventory = []
column_profiles = []

parquet_files = sorted(PROCESSED_DIR.rglob("*.parquet"))

for path in parquet_files:
    relative_path = path.relative_to(PROJECT_ROOT)

    try:
        metadata = pq.read_metadata(path)
        schema = pq.read_schema(path)

        inventory.append({
            "file": str(relative_path),
            "status": "OK",
            "rows": metadata.num_rows,
            "columns_count": metadata.num_columns,
            "size_mb": round(path.stat().st_size / 1024**2, 2),
            "error": None,
        })

        # Проверяем пропуски по каждому столбцу.
        df = pd.read_parquet(path)

        for column in df.columns:
            column_profiles.append({
                "file": str(relative_path),
                "column": column,
                "dtype": str(df[column].dtype),
                "null_count": int(df[column].isna().sum()),
                "null_pct": round(
                    df[column].isna().mean() * 100, 2
                ),
                "unique_count": int(df[column].nunique()),
            })

        print(
            f"OK | {relative_path} | "
            f"{metadata.num_rows:,} строк | "
            f"{metadata.num_columns} столбцов"
        )

    except Exception as exc:
        inventory.append({
            "file": str(relative_path),
            "status": "ERROR",
            "rows": None,
            "columns_count": None,
            "size_mb": round(path.stat().st_size / 1024**2, 2),
            "error": str(exc),
        })

        print(f"ERROR | {relative_path} | {exc}")

inventory_df = pd.DataFrame(inventory)
columns_df = pd.DataFrame(column_profiles)

inventory_path = REPORTS_DIR / "processed_inventory.csv"
profile_path = REPORTS_DIR / "processed_column_profile.csv"

inventory_df.to_csv(
    inventory_path, index=False, encoding="utf-8-sig"
)
columns_df.to_csv(
    profile_path, index=False, encoding="utf-8-sig"
)

print("\n--- ИТОГО ---")
print("Найдено Parquet:", len(parquet_files))
print("Успешно прочитано:", (inventory_df["status"] == "OK").sum())
print("Ошибок:", (inventory_df["status"] == "ERROR").sum())
print("Отчёт по файлам:", inventory_path)
print("Отчёт по столбцам:", profile_path)

display(inventory_df)


OK | processed\games\epic_games.parquet | 899 строк | 17 столбцов
OK | processed\games\rawg_games.parquet | 899,585 строк | 49 столбцов
OK | processed\games\steam_games.parquet | 144,499 строк | 43 столбцов
OK | processed\players\steam_monthly_players.parquet | 612,265 строк | 7 столбцов
OK | processed\twitch\twitch_games.parquet | 21,000 строк | 12 столбцов

--- ИТОГО ---
Найдено Parquet: 5
Успешно прочитано: 5
Ошибок: 0
Отчёт по файлам: C:\Users\245108\Desktop\gaming-data-analysis\docs\data_quality\processed_inventory.csv
Отчёт по столбцам: C:\Users\245108\Desktop\gaming-data-analysis\docs\data_quality\processed_column_profile.csv


,file,status,rows,columns_count,size_mb,error
0,processed\games\epic_games.parquet,OK,899,17,0.09,None
1,processed\games\rawg_games.parquet,OK,899585,49,788.52,None
2,processed\games\steam_games.parquet,OK,144499,43,234.56,None
3,processed\players\steam_monthly_players.parquet,OK,612265,7,5.90,None
4,processed\twitch\twitch_games.parquet,OK,21000,12,0.46,None


In [32]:

from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DIR = PROJECT_ROOT / "raw"
REPORTS_DIR = PROJECT_ROOT / "docs" / "data_quality"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

files = []

for path in RAW_DIR.rglob("*"):
    if not path.is_file():
        continue

    files.append({
        "source_file": str(path.relative_to(PROJECT_ROOT)),
        "filename": path.name,
        "extension": path.suffix.lower() or "[без расширения]",
        "size_mb": round(path.stat().st_size / 1024**2, 2),
        "folder": str(path.parent.relative_to(PROJECT_ROOT)),
    })

raw_inventory = pd.DataFrame(files)

if not raw_inventory.empty:
    raw_inventory = raw_inventory.sort_values(
        ["folder", "filename"]
    ).reset_index(drop=True)

raw_inventory.to_csv(
    REPORTS_DIR / "raw_inventory.csv",
    index=False,
    encoding="utf-8-sig",
)

print("Всего файлов в raw:", len(raw_inventory))

if not raw_inventory.empty:
    print("\nФайлы по расширениям:")
    display(
        raw_inventory.groupby("extension")
        .agg(
            files=("filename", "count"),
            total_size_mb=("size_mb", "sum"),
        )
        .reset_index()
        .sort_values("files", ascending=False)
    )

    print("\nПолный каталог:")
    display(raw_inventory)

print("\nОтчёт сохранён в docs/data_quality/raw_inventory.csv")


Всего файлов в raw: 42

Файлы по расширениям:


,extension,files,total_size_mb
0,.csv,36,9101.71
1,.json,3,942.02
2,.jsonl,1,4101.47
3,.parquet,1,964.80
4,.py,1,0.01



Полный каталог:


,source_file,filename,extension,size_mb,folder
0,raw\epic\epic games store-video games.csv,epic games store-video games.csv,.csv,0.25,raw\epic
1,raw\epic\epic-games-apps.csv,epic-games-apps.csv,.csv,0.00,raw\epic
2,raw\epic\epic-games-demo.csv,epic-games-demo.csv,.csv,0.01,raw\epic
3,raw\epic\epic_games_editor.csv,epic_games_editor.csv,.csv,0.01,raw\epic
4,raw\games\ultimate_games\Ultimate_Games_Dataset.csv,Ultimate_Games_Dataset.csv,.csv,27.54,raw\games\ultimate_games
5,raw\rawg\videogames_data.csv,videogames_data.csv,.csv,1907.80,raw\rawg
6,raw\rawg\videogames_data.parquet,videogames_data.parquet,.parquet,964.80,raw\rawg
7,raw\rawg\jsonl\videogames_data.jsonl,videogames_data.jsonl,.jsonl,4101.47,raw\rawg\jsonl
8,raw\steam\fronkongames\games.csv,games.csv,.csv,382.56,raw\steam\fronkongames
9,raw\steam\fronkongames\games.json,games.json,.json,925.28,raw\steam\fronkongames



Отчёт сохранён в docs/data_quality/raw_inventory.csv


In [33]:

from pathlib import Path
import duckdb
import pandas as pd

# Используем корень проекта, если он уже определён в ноутбуке.
if "PROJECT_ROOT" not in globals():
    PROJECT_ROOT = Path.cwd()
    if PROJECT_ROOT.name == "notebooks":
        PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DIR = PROJECT_ROOT / "raw"
REPORTS_DIR = PROJECT_ROOT / "docs" / "data_quality"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

con = duckdb.connect()
schema_rows = []
errors = []

for path in sorted(RAW_DIR.rglob("*")):
    if not path.is_file():
        continue

    ext = path.suffix.lower()
    if ext not in {".csv", ".json", ".jsonl", ".parquet"}:
        continue

    # SQL-пути корректно экранируются для Windows.
    sql_path = str(path.resolve()).replace("\\", "/").replace("'", "''")

    try:
        if ext == ".csv":
            source_sql = (
                f"read_csv_auto('{sql_path}', "
                "sample_size=10000, ignore_errors=true)"
            )
        elif ext in {".json", ".jsonl"}:
            source_sql = (
                f"read_json_auto('{sql_path}', "
                "sample_size=20480, ignore_errors=true)"
            )
        else:
            source_sql = f"read_parquet('{sql_path}')"

        description = con.execute(
            f"DESCRIBE SELECT * FROM {source_sql}"
        ).fetchdf()

        for _, column in description.iterrows():
            schema_rows.append({
                "source_file": str(path.relative_to(PROJECT_ROOT)),
                "column": column["column_name"],
                "data_type": column["column_type"],
                "extension": ext,
                "size_mb": round(path.stat().st_size / 1024**2, 2),
                "status": "ok",
                "error": None,
            })

        print(f"OK: {path.relative_to(PROJECT_ROOT)} "
              f"— {len(description)} колонок")

    except Exception as e:
        errors.append({
            "source_file": str(path.relative_to(PROJECT_ROOT)),
            "error": str(e)[:500],
        })
        print(f"ERROR: {path.name}: {str(e)[:200]}")

schema_df = pd.DataFrame(schema_rows)
schema_path = REPORTS_DIR / "source_schemas.csv"
schema_df.to_csv(schema_path, index=False, encoding="utf-8-sig")

errors_df = pd.DataFrame(errors)
errors_path = REPORTS_DIR / "schema_errors.csv"
errors_df.to_csv(errors_path, index=False, encoding="utf-8-sig")

print("\nПроверка завершена")
print("Файлов с определёнными схемами:",
      schema_df["source_file"].nunique() if not schema_df.empty else 0)
print("Колонок найдено:", len(schema_df))
print("Ошибок:", len(errors))
print("Схемы:", schema_path)
print("Ошибки:", errors_path)

if not schema_df.empty:
    display(
        schema_df.groupby("source_file")
        .agg(
            columns=("column", "count"),
            size_mb=("size_mb", "first"),
        )
        .reset_index()
        .sort_values("size_mb", ascending=False)
    )

con.close()


OK: raw\epic\epic games store-video games.csv — 17 колонок
OK: raw\epic\epic-games-apps.csv — 16 колонок
OK: raw\epic\epic-games-demo.csv — 17 колонок
OK: raw\epic\epic_games_editor.csv — 20 колонок
OK: raw\games\ultimate_games\Ultimate_Games_Dataset.csv — 43 колонок
OK: raw\rawg\jsonl\videogames_data.jsonl — 54 колонок
OK: raw\rawg\videogames_data.csv — 54 колонок
OK: raw\rawg\videogames_data.parquet — 54 колонок
OK: raw\steam\fronkongames\games.csv — 39 колонок
ERROR: games.json: Invalid Input Error: "maximum_object_size" of 16777216 bytes exceeded while reading file "C:/Users/245108/Desktop/gaming-data-analysis/raw/steam/fronkongames/games.json" (>33554428 bytes).
 Try increa
OK: raw\steam\games_2021_2025\a_steam_data_2021_2025.csv — 10 колонок
OK: raw\steam\games_2025\steam_dataset_2025_csv\application_categories.csv — 2 колонок
OK: raw\steam\games_2025\steam_dataset_2025_csv\application_developers.csv — 2 колонок
OK: raw\steam\games_2025\steam_dataset_2025_csv\application_genres.c

,source_file,columns,size_mb
5,raw\rawg\jsonl\videogames_data.jsonl,54,4101.47
36,raw\steam\reviews\dataset.csv,5,2060.77
34,raw\steam\recommendations\recommendations.csv,8,1929.50
6,raw\rawg\videogames_data.csv,54,1907.80
7,raw\rawg\videogames_data.parquet,54,964.80
22,raw\steam\games_2025\steam_dataset_2025_csv\reviews.csv,23,638.78
24,raw\steam\games_artemiloff\games_march2025_full.csv,47,449.65
23,raw\steam\games_artemiloff\games_march2025_cleaned.csv,47,446.93
26,raw\steam\games_artemiloff\games_may2024_full.csv,46,405.23
25,raw\steam\games_artemiloff\games_may2024_cleaned.csv,46,402.75


In [35]:

from pathlib import Path
import duckdb

steam_2025 = (
    RAW_DIR / "steam" / "games_2025" / "steam_dataset_2025_csv"
)
output_dir = PROJECT_ROOT / "processed" / "steam_2025"
output_dir.mkdir(parents=True, exist_ok=True)

con = duckdb.connect()

files_to_process = {
    "applications": "applications.csv",
    "application_categories": "application_categories.csv",
    "application_developers": "application_developers.csv",
    "application_genres": "application_genres.csv",
    "application_platforms": "application_platforms.csv",
    "application_publishers": "application_publishers.csv",
    "categories": "categories.csv",
    "developers": "developers.csv",
    "genres": "genres.csv",
    "platforms": "platforms.csv",
    "publishers": "publishers.csv",
}

for table_name, filename in files_to_process.items():
    source = (steam_2025 / filename).resolve()
    target = (output_dir / f"{table_name}.parquet").resolve()

    source_sql = str(source).replace("\\", "/").replace("'", "''")
    target_sql = str(target).replace("\\", "/").replace("'", "''")

    try:
        con.execute(f"""
            COPY (
                SELECT *
                FROM read_csv_auto(
                    '{source_sql}',
                    sample_size=-1,
                    ignore_errors=false
                )
            )
            TO '{target_sql}'
            (FORMAT PARQUET, COMPRESSION ZSTD)
        """)

        rows = con.execute(
            f"SELECT COUNT(*) FROM read_parquet('{target_sql}')"
        ).fetchone()[0]

        print(f"OK: {table_name}: {rows:,} строк")

    except Exception as e:
        print(f"ERROR: {table_name}: {e}")

con.close()


OK: applications: 239,664 строк
OK: application_categories: 1,077,156 строк
OK: application_developers: 250,817 строк
OK: application_genres: 587,515 строк
OK: application_platforms: 334,671 строк
OK: application_publishers: 223,048 строк
OK: categories: 462 строк
OK: developers: 101,226 строк
OK: genres: 154 строк
OK: platforms: 3 строк
OK: publishers: 85,699 строк


In [36]:

import json
from pathlib import Path

import ijson
import pyarrow as pa
import pyarrow.parquet as pq

source = (
    PROJECT_ROOT
    / "raw"
    / "steam"
    / "fronkongames"
    / "games.json"
)

output_dir = PROJECT_ROOT / "processed" / "games"
output_dir.mkdir(parents=True, exist_ok=True)

target = output_dir / "steam_games_json.parquet"
temp_target = output_dir / "steam_games_json.tmp.parquet"

# Удаляем только незавершённый результат предыдущего запуска.
for path in (target, temp_target):
    if path.exists():
        path.unlink()

def to_json_string(value):
    return json.dumps(
        value,
        ensure_ascii=False,
        default=str,
        separators=(",", ":"),
    )

def normalize_record(app_id, record, columns):
    row = {"app_id": str(app_id)}

    for column in columns:
        value = record.get(column)

        if isinstance(value, (dict, list)):
            row[column] = to_json_string(value)
        elif value is None:
            row[column] = None
        elif isinstance(value, bool):
            row[column] = str(value).lower()
        else:
            row[column] = str(value)

    return row

batch_size = 2000
rows = []
writer = None
total_rows = 0
columns = None

try:
    with source.open("rb") as file:
        # Каждый верхнеуровневый ключ — App ID,
        # значение — объект с данными игры.
        for app_id, record in ijson.kvitems(file, ""):
            if not isinstance(record, dict):
                continue

            # Фиксируем набор полей по первой записи.
            if columns is None:
                columns = sorted(record.keys())

            rows.append(
                normalize_record(app_id, record, columns)
            )

            if len(rows) >= batch_size:
                if writer is None:
                    schema = pa.schema(
                        [pa.field("app_id", pa.string())]
                        + [
                            pa.field(column, pa.string())
                            for column in columns
                        ]
                    )
                    writer = pq.ParquetWriter(
                        temp_target,
                        schema,
                        compression="zstd",
                    )

                table = pa.Table.from_pylist(
                    rows,
                    schema=writer.schema,
                )
                writer.write_table(table)

                total_rows += len(rows)
                rows.clear()

                if total_rows % 20000 == 0:
                    print(f"Обработано игр: {total_rows:,}")

    if rows:
        if writer is None:
            columns = sorted(rows[0].keys() - {"app_id"})
            schema = pa.schema(
                [pa.field("app_id", pa.string())]
                + [
                    pa.field(column, pa.string())
                    for column in columns
                ]
            )
            writer = pq.ParquetWriter(
                temp_target,
                schema,
                compression="zstd",
            )

        writer.write_table(
            pa.Table.from_pylist(rows, schema=writer.schema)
        )
        total_rows += len(rows)

    if writer is None:
        raise ValueError("В JSON не найдено ни одной записи игры.")

    writer.close()
    writer = None

    # Публикуем готовый результат только после успешной записи.
    temp_target.replace(target)

    print("\nГотово!")
    print("Файл:", target)
    print("Количество игр:", f"{total_rows:,}")
    print("Размер Parquet, MB:", round(target.stat().st_size / 1024**2, 2))

except Exception:
    if writer is not None:
        writer.close()
    if temp_target.exists():
        temp_target.unlink()
    raise


Обработано игр: 20,000
Обработано игр: 40,000
Обработано игр: 60,000
Обработано игр: 80,000
Обработано игр: 100,000
Обработано игр: 120,000
Обработано игр: 140,000

Готово!
Файл: C:\Users\245108\Desktop\gaming-data-analysis\processed\games\steam_games_json.parquet
Количество игр: 144,499
Размер Parquet, MB: 235.97


In [37]:
import duckdb

con = duckdb.connect()

display(con.execute(f"""
    SELECT app_id, name, release_date, price, genres
    FROM read_parquet('{str(target).replace(chr(92), '/')}')
    LIMIT 10
""").fetchdf())

print(
    "Всего строк:",
    con.execute(
        f"SELECT COUNT(*) FROM read_parquet('{str(target).replace(chr(92), '/')}')"
    ).fetchone()[0]
)

con.close()

,app_id,name,release_date,price,genres
0,2539430,Black Dragon Mage Playtest,"Aug 1, 2023",0.0,[]
1,496350,Supipara - Chapter 1 Spring Has Come!,"Jul 29, 2016",5.24,"[""Adventure""]"
2,1034400,Mystery Solitaire The Black Raven,"May 6, 2019",4.99,"[""Casual""]"
3,3292190,버튜버 파라노이아 - Vtuber Paranoia,"Oct 31, 2024",8.99,"[""Casual"",""Indie"",""Simulation""]"
4,3631080,Maze Quest VR,"Apr 24, 2025",4.99,"[""Action"",""Early Access""]"
5,1654170,Agony VR,"Apr 5, 2023",13.99,"[""Action"",""Adventure""]"
6,1934300,Armored Brigade II,"Apr 8, 2025",35.99,"[""Simulation"",""Strategy""]"
7,700340,Galacatraz: Eject Equip Escape,"Feb 1, 2018",0.99,"[""Action"",""Indie""]"
8,1157670,Hepta Beats,"May 7, 2021",0.99,"[""Casual"",""Indie""]"
9,1540330,MUMBA IV: Egypt Jewels,"Dec 13, 2021",0.59,"[""Action"",""Casual"",""Indie"",""Strategy""]"


Всего строк: 144499


In [40]:

from pathlib import Path
import duckdb

PROJECT_ROOT = Path(PROJECT_ROOT)
RAW_DIR = PROJECT_ROOT / "raw"
PROCESSED_DIR = PROJECT_ROOT / "processed"

jobs = [
    (
        RAW_DIR / "twitch" / "evolution" / "Twitch_global_data.csv",
        PROCESSED_DIR / "twitch" / "twitch_global.parquet",
    ),
    (
        RAW_DIR / "twitch" / "streamers" / "datasetV2.csv",
        PROCESSED_DIR / "twitch" / "twitch_streamers.parquet",
    ),
    (
        RAW_DIR / "steam" / "games_2025" / "steam_dataset_2025_csv" / "reviews.csv",
        PROCESSED_DIR / "reviews" / "steam_2025_reviews.parquet",
    ),
]

con = duckdb.connect()
results = []

for source, target in jobs:
    if not source.exists():
        print(f"SKIP: исходный файл не найден: {source}")
        continue

    target.parent.mkdir(parents=True, exist_ok=True)
    source_sql = str(source.resolve()).replace("\\", "/").replace("'", "''")
    target_sql = str(target.resolve()).replace("\\", "/").replace("'", "''")

    try:
        con.execute(f"""
            COPY (
                SELECT *
                FROM read_csv_auto(
                    '{source_sql}',
                    sample_size=-1,
                    ignore_errors=false
                )
            )
            TO '{target_sql}'
            (FORMAT PARQUET, COMPRESSION ZSTD)
        """)

        rows = con.execute(
            f"SELECT COUNT(*) FROM read_parquet('{target_sql}')"
        ).fetchone()[0]

        size_mb = round(target.stat().st_size / 1024**2, 2)
        results.append({
            "source": str(source.relative_to(PROJECT_ROOT)),
            "output": str(target.relative_to(PROJECT_ROOT)),
            "rows": rows,
            "size_mb": size_mb,
            "status": "OK",
        })
        print(f"OK: {target.name} — {rows:,} строк, {size_mb} MB")

    except Exception as e:
        results.append({
            "source": str(source.relative_to(PROJECT_ROOT)),
            "output": str(target.relative_to(PROJECT_ROOT)),
            "rows": None,
            "size_mb": None,
            "status": str(e)[:300],
        })
        print(f"ERROR: {source.name}: {e}")

con.close()

import pandas as pd
results_df = pd.DataFrame(results)
display(results_df)

report_dir = PROJECT_ROOT / "docs" / "data_quality"
report_dir.mkdir(parents=True, exist_ok=True)
results_df.to_csv(
    report_dir / "additional_processing_report.csv",
    index=False,
    encoding="utf-8-sig",
)


OK: twitch_global.parquet — 105 строк, 0.0 MB
OK: twitch_streamers.parquet — 999 строк, 0.04 MB
OK: steam_2025_reviews.parquet — 1,048,148 строк, 222.98 MB


,source,output,rows,size_mb,status
0,raw\twitch\evolution\Twitch_global_data.csv,processed\twitch\twitch_global.parquet,105,0.00,OK
1,raw\twitch\streamers\datasetV2.csv,processed\twitch\twitch_streamers.parquet,999,0.04,OK
2,raw\steam\games_2025\steam_dataset_2025_csv\reviews.csv,processed\reviews\steam_2025_reviews.parquet,1048148,222.98,OK
